In [25]:
import zipfile

zip_path = "archive.zip"
extract_path = "."

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [15]:
df = spark.read.csv(f"{extract_path}/accepted_2007_to_2018q4.csv", header=True, inferSchema=True)
df.show(5)

+--------+---------+---------+-----------+---------------+----------+--------+-----------+-----+---------+--------------------+----------+--------------+----------+-------------------+--------+-----------+----------+--------------------+----+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+---

In [26]:
from pyspark.sql.functions import substring, col

df_with_year = df.withColumn("year", substring(col("issue_d"), -4, 4))

df_with_year.select("year").distinct().orderBy("year").show()

+----+
|year|
+----+
|null|
|2007|
|2008|
|2009|
|2010|
|2011|
|2012|
|2013|
|2014|
|2015|
|2016|
|2017|
|2018|
|fied|
+----+



In [28]:
from pyspark.sql.functions import col, substring, when

df_accepted = df_accepted.withColumn("year", substring(col("issue_d"), -4, 4).cast("int"))

df_rejected = df_rejected.withColumn("year", substring(col("application_date"), -4, 4).cast("int"))

df_accepted = df_accepted.filter(col("year").isNotNull())
df_rejected = df_rejected.filter(col("year").isNotNull())

year_bucket_expr = when(col("year") <= 2016, "2007-2016") \
                   .when(col("year") == 2017, "2017") \
                   .otherwise("2018")

df_accepted = df_accepted.withColumn("year_bucket", year_bucket_expr)
df_rejected = df_rejected.withColumn("year_bucket", year_bucket_expr)

df_accepted.write.mode("overwrite").partitionBy("year_bucket").parquet("accepted")
df_rejected.write.mode("overwrite").partitionBy("year_bucket").parquet("rejected")


In [30]:
from pyspark.sql.functions import col, substring

df_accepted = df_accepted.withColumn("year", substring(col("issue_d"), -4, 4).cast("int"))

df_accepted = df_accepted.filter(col("year").isNotNull())

df_accepted.write.mode("overwrite").partitionBy("year").parquet("accepted_by_year")


df_rejected = df_rejected.withColumn("year", substring(col("application_date"), -4, 4).cast("int"))
df_rejected = df_rejected.filter(col("year").isNotNull())
df_rejected.write.mode("overwrite").partitionBy("year").parquet("rejected_by_year")
